# PART 2 — 数値はどうやって予測する？
## 実習1：糖尿病の進行度を予測しよう

**Features → Linear Regression → Numerical Prediction**

この実習では、scikit-learn に用意されている **Diabetes Dataset** を使って、
患者の特徴量から **1年後の糖尿病の進行度を表す数値** を予測します。

今回は、回帰の基本モデルである **線形回帰（Linear Regression）** を使います。

### この実習で学ぶこと
1. 入力 **X** と正解 **y** を確認する
2. データを学習用とテスト用に分ける
3. 線形回帰モデルを学習する
4. テストデータを予測する
5. **MAE / MSE / RMSE / R²** で評価する
6. 実際の値と予測値をグラフで確認する

> このデータセットは機械学習の学習用データです。  
> 実際の医療診断を行うためのプログラムではありません。


## 1. ライブラリを読み込む

まず、今回使うライブラリを読み込みます。

- `pandas`：表形式のデータを扱う
- `numpy`：数値計算
- `matplotlib`：グラフを描く
- `scikit-learn`：データセット、機械学習モデル、評価指標を使う


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Diabetes Dataset を読み込む

`load_diabetes()` を使うと、データをインターネットから別にダウンロードしなくても利用できます。

今回は `as_frame=True` を指定して、最初から pandas の表として読み込みます。


In [ ]:
# Diabetes Dataset を読み込む
diabetes = load_diabetes(as_frame=True)

# 特徴量と target をまとめた表
df = diabetes.frame.copy()

print("行数と列数:", df.shape)
display(df.head())


## 3. データの内容を確認する

このデータには **442人分のデータ** があり、入力には **10個の特徴量** を使います。

| 特徴量 | 内容 |
|---|---|
| age | 年齢 |
| sex | 性別 |
| bmi | BMI（体格指数） |
| bp | 血圧 |
| s1 ～ s6 | 6種類の血液検査に関する値 |

これらの特徴量は、あらかじめスケーリングされた数値です。

`target` は、**1年後の糖尿病の進行度を表す数値**です。

### ここで重要
- **X**：10個の特徴量
- **y**：糖尿病の進行度を表す数値

分類では `y` がクラスでしたが、  
**回帰では `y` が連続した数値**になります。


In [ ]:
print("列名:")
print(df.columns.tolist())

print("\nデータ数:", len(df))
print("欠損値の数:", df.isnull().sum().sum())


## 4. 入力 X と正解 y を分ける

`target` 以外の10列を入力 **X** にします。

`target` 列を正解 **y** にします。


In [ ]:
# 入力 X：10個の特徴量
X = df.drop(columns=["target"])

# 正解 y：糖尿病の進行度
y = df["target"]

print("X の形:", X.shape)
print("y の形:", y.shape)

display(X.head())
display(y.head())


## 5. 学習用データとテスト用データに分ける

データの **80%を学習用（Train）**、**20%をテスト用（Test）** に分けます。

- 学習用データ：モデルを学習するために使う
- テスト用データ：学習には使わず、最後に予測性能を確認するために使う

`random_state=42` を指定すると、毎回同じ分け方になります。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("学習用データ:", len(X_train))
print("テスト用データ:", len(X_test))


## 6. 線形回帰モデルを学習する

`LinearRegression()` で線形回帰モデルを作ります。

`fit()` に学習用の **X** と **y** を渡すと、モデルがデータの関係を学習します。


In [ ]:
# 線形回帰モデルを作る
model = LinearRegression()

# 学習
model.fit(X_train, y_train)

print("学習が完了しました。")


## 7. テストデータを予測する

学習が終わったモデルに、学習には使っていない `X_test` を入力します。

`predict()` の結果が **予測値** です。


In [ ]:
# テストデータを予測
y_pred = model.predict(X_test)

print("予測したデータ数:", len(y_pred))
print("最初の5個の予測値:")
print(np.round(y_pred[:5], 1))


## 8. 実際の値と予測値を比べる

予測した値と、本当の正解を並べて確認してみましょう。

完全に同じ値になる必要はありません。  
**どのくらい近い値を予測できたか**を見ることが回帰では重要です。


In [ ]:
result = pd.DataFrame({
    "実際の値": y_test.values,
    "予測値": y_pred
})

result["誤差"] = result["実際の値"] - result["予測値"]

display(result.head(10).round(1))


## 9. 回帰モデルを評価する

今回は4つの指標を確認します。

- **MAE（Mean Absolute Error）**：平均絶対誤差
- **MSE（Mean Squared Error）**：平均二乗誤差
- **RMSE（Root Mean Squared Error）**：二乗平均平方根誤差
- **R²（Coefficient of Determination）**：決定係数

### 見方
- MAE：小さいほどよい
- MSE：小さいほどよい
- RMSE：小さいほどよい
- R²：一般的には 1 に近いほどよい

MAE と RMSE は、`y` と同じ単位で考えることができます。


In [ ]:
# MAE
mae = mean_absolute_error(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE = MSE の平方根
rmse = np.sqrt(mse)

# R^2
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))


## 10. 実際の値と予測値をグラフで確認する

横軸を **実際の値**、縦軸を **予測値** にします。

点が赤い対角線に近いほど、実際の値に近い予測ができています。


In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(y_test, y_pred, alpha=0.7)

# 実際の値 = 予測値 となる基準線
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--",
    label="Actual = Predicted"
)

plt.xlabel("Actual value")
plt.ylabel("Predicted value")
plt.title("Actual vs Predicted")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 11. 残差を確認する

**残差（Residual）** は、

**実際の値 − 予測値**

です。

残差が 0 に近いほど、そのデータに対する予測が実際の値に近かったことを意味します。


In [ ]:
residuals = y_test.values - y_pred

residual_df = pd.DataFrame({
    "実際の値": y_test.values,
    "予測値": y_pred,
    "残差": residuals
})

display(residual_df.head(10).round(1))

print("残差の平均:", round(residuals.mean(), 2))


## 12. 学習に使わなかった1人分を予測してみる

テストデータから1人分だけ取り出して、モデルに予測させます。

これは新しく入力した患者ではなく、  
**学習には使わなかったテストデータの1件**です。


In [ ]:
# テストデータから1行だけ取り出す
one_data = X_test.iloc[[0]]

# 予測
one_prediction = model.predict(one_data)

print("予測値:", round(one_prediction[0], 1))
print("実際の値:", y_test.iloc[0])


# まとめ

今回の回帰実習では、次の流れを確認しました。

### 回帰の基本手順

**1. データを確認する**  
入力 X と正解 y は何か？

↓

**2. データを分割する**  
学習用とテスト用に分ける

↓

**3. モデルを学習する**  
`LinearRegression()` と `fit()`

↓

**4. 予測する**  
`predict()`

↓

**5. 評価する**  
MAE / MSE / RMSE / R²

### 分類との違い

- **分類**：クラス・種類を予測する
- **回帰**：連続した数値を予測する

データが変わっても、  
**確認 → 分割 → 学習 → 予測 → 評価**  
という基本的な流れは共通です。


# 確認問題

次の問いに答えてください。

1. 今回の入力 **X** は何ですか。
2. 今回の正解 **y** は何ですか。
3. 今回は分類と回帰のどちらの問題ですか。その理由も書いてください。
4. 学習用データとテスト用データは何対何に分けましたか。
5. 今回得られた **MAE、MSE、RMSE、R²** の値を書いてください。
6. MAE、MSE、RMSE は、大きい方と小さい方のどちらがよいですか。
7. R² は、一般的にどの値に近いほどよいですか。
8. 「実際の値と予測値」のグラフを見て、予測結果について気づいたことを1つ書いてください。

### 提出するもの
- MAE / MSE / RMSE / R²
- Actual vs Predicted のグラフ
- 上の確認問題への回答


## 参考

- scikit-learn: Diabetes Dataset
- scikit-learn: LinearRegression
- scikit-learn: Regression metrics

この実習では、機械学習の基本的な流れを理解することを目的として、
複雑な前処理やモデル選択は行っていません。
